In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 02 - Analytical Track Development and Validation (POC-41)\n",
    "Track A: Comparative Intelligence."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "from pathlib import Path\n",
    "\n",
    "# Robust path resolution\n",
    "ROOT = Path.cwd().resolve().parents[1]\n",
    "DATA_PATH = ROOT / 'data' / 'canonical' / 'intelligence_data.csv'\n",
    "\n",
    "df = pd.read_csv(DATA_PATH, dtype={'is_synthetic': str})\n",
    "fe_data = df[(df['source_name'].str.startswith('Frontend')) & (df['record_type'] == 'measurement')].copy()\n",
    "\n",
    "# Baseline Calculation\n",
    "baseline = fe_data.groupby('metric_name')['metric_value'].mean()\n",
    "print('Baseline:', baseline.to_dict())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "grouped = fe_data.groupby(['entity_id', 'subcategory', 'category', 'metric_name']).agg(\n",
    "    record_count=('metric_value', 'count'),\n",
    "    avg_value=('metric_value', 'mean')\n",
    ").reset_index()\n",
    "\n",
    "pivot = grouped.pivot_table(index=['entity_id', 'subcategory', 'category', 'record_count'], columns='metric_name', values='avg_value').reset_index()\n",
    "\n",
    "pivot['output_gap_vs_baseline'] = pivot['output_per_hour'] - baseline['output_per_hour']\n",
    "pivot['overtime_gap_vs_baseline'] = pivot['overtime_hours'] - baseline['overtime_hours']\n",
    "\n",
    "pivot['priority_rank'] = pivot['output_gap_vs_baseline'].rank(method='dense', ascending=True).astype(int)\n",
    "print(pivot.sort_values('priority_rank').to_string())"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}